# Day 55 — Custom Training Loops, TF Functions & Graphs
- Understand manual/custom training loops
- Implement forward pass, loss, gradients, and optimizer updates
- Track metrics manually
- Understand `tf.function`
- Understand tracing and TensorFlow graphs
- Understand AutoGraph and TensorFlow-compatible control flow
- Understand common `tf.function` rules and retracing


In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np

print("TensorFlow:", tf.__version__)


TensorFlow: 2.21.0


## 1. The standard Keras training loop

Usually Keras hides the training loop:

```python
model.fit(X_train, y_train, epochs=10)
```

Internally, training repeatedly performs:

1. Forward pass
2. Loss calculation
3. Gradient calculation
4. Weight update
5. Metric update

Today we will implement these steps ourselves.


In [2]:
# Create a small regression dataset
np.random.seed(42)

X = np.random.randn(1000, 1).astype(np.float32)
y = (3 * X + 2 + 0.5 * np.random.randn(1000, 1)).astype(np.float32)

train_dataset = tf.data.Dataset.from_tensor_slices((X, y))
train_dataset = train_dataset.shuffle(1000).batch(32)

model = keras.Sequential([
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1)
])

loss_fn = keras.losses.MeanSquaredError()
optimizer = keras.optimizers.SGD(learning_rate=0.01)

print(model.summary())


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


## 2. Manual training loop

The central training pattern is:

```text
inputs
  ↓
model prediction
  ↓
loss
  ↓
GradientTape
  ↓
gradients
  ↓
optimizer
  ↓
updated weights
```


In [3]:
epochs = 5

for epoch in range(epochs):
    epoch_loss = []

    for X_batch, y_batch in train_dataset:
        # Forward pass + loss
        with tf.GradientTape() as tape:
            y_pred = model(X_batch, training=True)
            loss = loss_fn(y_batch, y_pred)

        # Calculate gradients
        gradients = tape.gradient(
            loss,
            model.trainable_variables
        )

        # Update weights
        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        epoch_loss.append(float(loss))

    print(
        f"Epoch {epoch + 1}/{epochs} - "
        f"loss: {np.mean(epoch_loss):.4f}"
    )


Epoch 1/5 - loss: 3.0728
Epoch 2/5 - loss: 0.5423
Epoch 3/5 - loss: 0.4337
Epoch 4/5 - loss: 0.3616
Epoch 5/5 - loss: 0.3448


## 3. Why `GradientTape`?

`tf.GradientTape()` records TensorFlow operations so TensorFlow can calculate derivatives.

For a loss:

```python
with tf.GradientTape() as tape:
    y_pred = model(X)
    loss = loss_fn(y, y_pred)

gradients = tape.gradient(
    loss,
    model.trainable_variables
)
```

Think:

**GradientTape → calculates gradients**

**Optimizer → uses gradients to update weights**


## 4. Put one training step into a function

Separating the training step makes the loop easier to understand and customize.


In [4]:
def train_step(model, X_batch, y_batch, loss_fn, optimizer):
    with tf.GradientTape() as tape:
        y_pred = model(X_batch, training=True)
        loss = loss_fn(y_batch, y_pred)

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    return loss


In [5]:
for epoch in range(3):
    losses = []

    for X_batch, y_batch in train_dataset:
        loss = train_step(
            model,
            X_batch,
            y_batch,
            loss_fn,
            optimizer
        )
        losses.append(float(loss))

    print(f"Epoch {epoch + 1}: loss = {np.mean(losses):.4f}")


Epoch 1: loss = 0.3186
Epoch 2: loss = 0.3101
Epoch 3: loss = 0.3032


## 5. Metrics in a custom training loop

Keras metric objects maintain state across batches.

Typical pattern:

```python
metric.update_state(...)
metric.result()
metric.reset_state()
```


In [6]:
mae_metric = keras.metrics.MeanAbsoluteError()

for X_batch, y_batch in train_dataset:
    y_pred = model(X_batch, training=False)
    mae_metric.update_state(y_batch, y_pred)

print("MAE:", float(mae_metric.result()))

mae_metric.reset_state()


MAE: 0.4368678033351898


## 6. A custom training loop with metrics

Here we combine the pieces into a more realistic loop.


In [7]:
def train_one_epoch(model, dataset, loss_fn, optimizer):
    loss_metric = keras.metrics.Mean()
    mae_metric = keras.metrics.MeanAbsoluteError()

    for X_batch, y_batch in dataset:
        with tf.GradientTape() as tape:
            y_pred = model(X_batch, training=True)
            loss = loss_fn(y_batch, y_pred)

        gradients = tape.gradient(
            loss,
            model.trainable_variables
        )

        optimizer.apply_gradients(
            zip(gradients, model.trainable_variables)
        )

        loss_metric.update_state(loss)
        mae_metric.update_state(y_batch, y_pred)

    return float(loss_metric.result()), float(mae_metric.result())


for epoch in range(3):
    loss_value, mae_value = train_one_epoch(
        model,
        train_dataset,
        loss_fn,
        optimizer
    )

    print(
        f"Epoch {epoch + 1}: "
        f"loss={loss_value:.4f}, "
        f"MAE={mae_value:.4f}"
    )


Epoch 1: loss=0.2921, MAE=0.4326
Epoch 2: loss=0.2853, MAE=0.4260
Epoch 3: loss=0.2766, MAE=0.4202


# Part 2 — TensorFlow Functions and Graphs

## 7. What is `tf.function`?

`@tf.function` asks TensorFlow to transform a Python function into a TensorFlow graph when possible.

Conceptually:

```text
Python function
      ↓
   tracing
      ↓
TensorFlow graph
      ↓
graph execution
```

This is especially useful for repeated numerical computations.


In [8]:
def add_python(a, b):
    return a + b


@tf.function
def add_tf(a, b):
    return a + b


a = tf.constant(10.0)
b = tf.constant(5.0)

print("Python function:", add_python(a, b).numpy())
print("tf.function:", add_tf(a, b).numpy())


Python function: 15.0
tf.function: 15.0


## 8. Inspecting a concrete function

A `tf.function` can create a concrete graph for a particular input signature.


In [9]:
concrete_add = add_tf.get_concrete_function(
    tf.TensorSpec(shape=(), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.float32)
)

print("Concrete function:")
print(concrete_add)

print("\nGraph operations:")
for operation in concrete_add.graph.get_operations():
    print(operation.name, "->", operation.type)


Concrete function:
ConcreteFunction Input Parameters:
  a (POSITIONAL_OR_KEYWORD): TensorSpec(shape=(), dtype=tf.float32, name=None)
  b (POSITIONAL_OR_KEYWORD): TensorSpec(shape=(), dtype=tf.float32, name=None)
Output Type:
  TensorSpec(shape=(), dtype=tf.float32, name=None)
Captures:
  None

Graph operations:
a -> Placeholder
b -> Placeholder
add -> AddV2
Identity -> Identity


## 9. Tracing

When TensorFlow first needs a compatible graph for a `tf.function`, it traces the Python code.

A useful mental model:

```text
First compatible call
       ↓
     trace
       ↓
 build TensorFlow graph
       ↓
 execute graph

Later compatible calls
       ↓
 reuse graph
```

Different input signatures can require additional tracing.


In [10]:
@tf.function
def square(x):
    return x ** 2

print(square(tf.constant([1., 2., 3.])).numpy())
print(square(tf.constant([4., 5., 6.])).numpy())


[1. 4. 9.]
[16. 25. 36.]


## 10. Input signatures

An input signature can make the expected input shape and dtype explicit.

`None` means a variable-sized dimension.


In [11]:
@tf.function(
    input_signature=[
        tf.TensorSpec(shape=[None], dtype=tf.float32)
    ]
)
def square_vector(x):
    return x ** 2


print(square_vector(tf.constant([1., 2., 3.])).numpy())
print(square_vector(tf.constant([10., 20., 30., 40.])).numpy())


[1. 4. 9.]
[ 100.  400.  900. 1600.]


## 11. AutoGraph

AutoGraph can transform suitable Python control-flow statements into TensorFlow-compatible graph operations.

For example, tensor-dependent conditions can be written using ordinary-looking Python syntax inside a `tf.function`.


In [12]:
@tf.function
def absolute_value(x):
    if x >= 0:
        return x
    else:
        return -x


print("absolute_value(5):", absolute_value(tf.constant(5)).numpy())
print("absolute_value(-5):", absolute_value(tf.constant(-5)).numpy())


absolute_value(5): 5
absolute_value(-5): 5


### TensorFlow control flow

For more explicit graph-level control flow, TensorFlow also provides operations such as:

- `tf.cond()`
- `tf.while_loop()`

AutoGraph often lets us write simpler Python-like code while TensorFlow handles the conversion.


In [13]:
@tf.function
def absolute_value_explicit(x):
    return tf.cond(
        x >= 0,
        lambda: x,
        lambda: -x
    )


print(absolute_value_explicit(tf.constant(-7)).numpy())


7


## 12. `tf.print()` vs Python `print()`

Inside graph-based TensorFlow code, prefer `tf.print()` when you want output as part of TensorFlow execution.

Python `print()` can execute during tracing rather than every graph execution.


In [14]:
@tf.function
def demonstrate_print(x):
    print("Python print: tracing/executing Python code")
    tf.print("tf.print: TensorFlow execution, x =", x)
    return x * 2


print("First call:")
print(demonstrate_print(tf.constant(3)).numpy())

print("\nSecond call:")
print(demonstrate_print(tf.constant(4)).numpy())


First call:
Python print: tracing/executing Python code
tf.print: TensorFlow execution, x = 3
6

Second call:
tf.print: TensorFlow execution, x = 4
8


## 13. `tf.function` and Python side effects

Be careful with Python state such as lists, counters, and mutable objects inside `tf.function`.

Prefer TensorFlow state such as `tf.Variable` when state needs to participate in repeated graph execution.


In [15]:
counter = tf.Variable(0, dtype=tf.int32, trainable=False)

@tf.function
def increment_counter():
    counter.assign_add(1)
    return counter


print(increment_counter().numpy())
print(increment_counter().numpy())
print(increment_counter().numpy())


1
2
3


## 14. Custom training step with `@tf.function`

Now combine the two main topics of Day 55.

The training step uses:

- `GradientTape`
- loss function
- gradients
- optimizer
- `tf.function`


In [16]:
@tf.function
def tf_train_step(model, X_batch, y_batch, loss_fn, optimizer):
    with tf.GradientTape() as tape:
        y_pred = model(X_batch, training=True)
        loss = loss_fn(y_batch, y_pred)

    gradients = tape.gradient(
        loss,
        model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, model.trainable_variables)
    )

    return loss


In [17]:
# Run the graph-based training step
for epoch in range(3):
    losses = []

    for X_batch, y_batch in train_dataset:
        loss = tf_train_step(
            model,
            X_batch,
            y_batch,
            loss_fn,
            optimizer
        )
        losses.append(float(loss))

    print(f"Epoch {epoch + 1}: loss = {np.mean(losses):.4f}")


Epoch 1: loss = 0.2685
Epoch 2: loss = 0.2634
Epoch 3: loss = 0.2666


# 15. Full custom training loop

This version keeps epoch-level loss and MAE metrics while using a graph-compiled training step.


In [18]:
loss_fn = keras.losses.MeanSquaredError()
optimizer = keras.optimizers.Adam(learning_rate=0.01)

# Fresh model for the final example
final_model = keras.Sequential([
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1)
])


@tf.function
def final_train_step(X_batch, y_batch):
    with tf.GradientTape() as tape:
        y_pred = final_model(X_batch, training=True)
        loss = loss_fn(y_batch, y_pred)

    gradients = tape.gradient(
        loss,
        final_model.trainable_variables
    )

    optimizer.apply_gradients(
        zip(gradients, final_model.trainable_variables)
    )

    return loss, y_pred


for epoch in range(5):
    loss_metric = keras.metrics.Mean()
    mae_metric = keras.metrics.MeanAbsoluteError()

    for X_batch, y_batch in train_dataset:
        loss, y_pred = final_train_step(X_batch, y_batch)

        loss_metric.update_state(loss)
        mae_metric.update_state(y_batch, y_pred)

    print(
        f"Epoch {epoch + 1}: "
        f"loss={loss_metric.result():.4f}, "
        f"MAE={mae_metric.result():.4f}"
    )


Epoch 1: loss=9.2873, MAE=2.3562
Epoch 2: loss=2.0275, MAE=0.9722
Epoch 3: loss=0.4173, MAE=0.5136
Epoch 4: loss=0.3089, MAE=0.4475
Epoch 5: loss=0.2787, MAE=0.4244
